#CLONE in Databricks?

CLONE is a Delta Lake feature that lets you create copies of Delta tables quickly without physically duplicating all the data immediately.

There are two types:

🪶 SHALLOW CLONE => meta data and refer old data (only at that point of time is copied and we can insert record in this table too from that time old refered data+ new data will be added)

💾 FULL CLONE => both meta data and data

🪶 1. SHALLOW CLONE
🔸 Definition:

Creates a new table metadata that references the same data files as the source table.

CREATE TABLE sales_clone
SHALLOW CLONE sales;

🔸 Key Points:
Property	Description
Data files	Shared with source table
Metadata	New copy (independent)
Storage cost	Very low
Speed	Very fast
Use case	Testing, experimentation, dev/test environments

👉 Only metadata is copied; no duplication of actual Parquet data.

🔸 Example:
-- Original table
CREATE TABLE sales (
  id INT, product STRING, amount DOUBLE
) USING delta;

INSERT INTO sales VALUES (1, 'Laptop', 90000), (2, 'Phone', 50000);

-- Shallow clone
CREATE TABLE sales_clone SHALLOW CLONE sales;

-- Both share same data files
DESCRIBE HISTORY sales;
DESCRIBE HISTORY sales_clone;


✅ Both tables will show similar file references initially.
✅ You can modify sales_clone independently later.

💾 2. DEEP CLONE
🔸 Definition:

Creates a full physical copy of both data and metadata of the source table.

CREATE TABLE sales_backup
DEEP CLONE sales;

🔸 Key Points:
| Property     | Description                                            |
| ------------ | ------------------------------------------------------ |
| Data files   | Fully copied                                           |
| Metadata     | New copy                                               |
| Storage cost | Same as original                                       |
| Speed        | Slower (full data copy)                                |
| Use case     | Backups, migrations, archiving, DR (disaster recovery) |


✅ Deep clone ensures data independence — future changes to the source table won’t affect the clone.

| Action                 | SHALLOW CLONE | DEEP CLONE |
| ---------------------- | ------------- | ---------- |
| Copy speed             | ⚡ Fast        | 🐢 Slower  |
| Data storage           | 🔗 Shared     | 📦 Copied  |
| Independent metadata   | ✅ Yes         | ✅ Yes      |
| Safe if source deleted | ❌ No          | ✅ Yes      |
| Common use             | Testing, QA   | Backup, DR |



In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_practice.inputdb.employees_sc (
  id INT,
  name STRING,
  department STRING,
  salary DECIMAL(10, 2),
  hire_date DATE
) USING DELTA;


In [0]:
%sql
INSERT INTO databricks_practice.inputdb.employees_sc VALUES
  (1, 'Alice Smith', 'Engineering', 75000.00, '2022-01-15'),
  (2, 'Bob Johnson', 'Marketing', 60000.00, '2021-03-20'),
  (3, 'Charlie Brown', 'Sales', 80000.00, '2023-06-01'),
  (4, 'Diana Prince', 'Engineering', 90000.00, '2020-11-10');



In [0]:
%sql
CREATE TABLE databricks_practice.inputdb.employees_sc_cp
SHALLOW CLONE databricks_practice.inputdb.employees_sc;

In [0]:
%sql
DESCRIBE DETAIL databricks_practice.inputdb.employees_sc_cp

In [0]:
%sql
select * from databricks_practice.inputdb.employees_sc_cp

In [0]:
    dbutils.fs.ls("databricks_practice.inputdb.employees_sc_cp")

In [0]:
%sql
INSERT INTO databricks_practice.inputdb.employees_sc VALUES
  (1, 'Alice Smith', 'Engineering', 75000.00, '2022-01-15'),
  (2, 'Bob Johnson', 'Marketing', 60000.00, '2021-03-20'),
  (3, 'Charlie Brown', 'Sales', 80000.00, '2023-06-01'),
  (4, 'Diana Prince', 'Engineering', 90000.00, '2020-11-10');



In [0]:
%sql
select * from databricks_practice.inputdb.employees_sc_cp

In [0]:
%sql
INSERT INTO databricks_practice.inputdb.employees_sc_cp VALUES
  (1, 'Alice Smith', 'Engineering', 75000.00, '2022-01-15'),
  (2, 'Bob Johnson', 'Marketing', 60000.00, '2021-03-20'),
  (3, 'Charlie Brown', 'Sales', 80000.00, '2023-06-01'),
  (4, 'Diana Prince', 'Engineering', 90000.00, '2020-11-10');



In [0]:
%sql
select * from databricks_practice.inputdb.employees_sc_cp

In [0]:
%sql
DESCRIBE DETAIL databricks_practice.inputdb.employees_sc_cp